In [1]:
pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝

import pandas as pd

csv_url = "https://raw.githubusercontent.com/tezamo/FPMA/main/wheat/wheat.csv"

df = pd.read_csv(csv_url, parse_dates=["date"], dayfirst=True)

# Display the first rows
df.head()

HTTPError: HTTP Error 404: Not Found

In [ ]:
# --------------------------------------------------
# 3. Create a unified market identifier
# --------------------------------------------------
df["effective_market"] = np.where(
    df["price_source"] == "International",
    df["market"],
    df["country"] + "_" + df["market"]
)

# --------------------------------------------------
# 4. Define independent time-series groups
# --------------------------------------------------
group_cols = [
    "commodity_name",
    "commodity_type",
    "price_source",
    "effective_market",
    "price_type",
    "unit_std"
]

# --------------------------------------------------
# 5. Sort data
# --------------------------------------------------
df = df.sort_values(group_cols + ["date"])

# --------------------------------------------------
# 6. Cubic spline interpolation function
# --------------------------------------------------
def spline_interpolate(group):
    group = group.set_index("date")

    # Require enough known points for cubic spline
    if group["price_per_unit"].notna().sum() >= 4:
        group["price_per_unit"] = group["price_per_unit"].interpolate(
            method="spline",
            order=3
        )

    return group.reset_index()

# --------------------------------------------------
# 7. Apply interpolation per independent series
# --------------------------------------------------
df_interpolated = (
    df
    .groupby(group_cols, group_keys=False)
    .apply(spline_interpolate)
)

# --------------------------------------------------
# 8. Save to a NEW file
# --------------------------------------------------
output_path = "https://raw.githubusercontent.com/tezamo/FPMA/main/wheat/wheat_interpolated.csv"
df_interpolated.to_csv(output_path, index=False)

print(f"File saved to: {output_path}")


In [ ]:
# How many values were filled?
filled_count = (
    df["price_per_unit"].isna().sum()
    - df_interpolated["price_per_unit"].isna().sum()
)

print(f"Number of filled values: {filled_count}")